In [4]:
from __future__ import annotations
from typing import Annotated, List, Dict, Optional, Literal
from typing_extensions import TypedDict

from pathlib import Path
from pydantic import BaseModel, Field

# LangGraph
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.types import Command

# LangChain
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, BaseMessage, SystemMessage
from langchain_core.tools import tool

# Tools
from langchain_tavily import TavilySearch
from langchain_community.document_loaders import WebBaseLoader
from langchain_experimental.utilities import PythonREPL

from dotenv import load_dotenv

load_dotenv()

# 1) 공용 상태 정의
class State(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    next: str

C:\Users\HKIT\AppData\Local\Temp\ipykernel_9940\2130707284.py:20: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.
C:\Users\HKIT\AppData\Local\Temp\ipykernel_9940\2130707284.py:21: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.utilities import PythonREPL


In [5]:
# 2) 리서치팀 도구
# Tavily 웹 검색: 상위 n개 결과를 반환
tavily_tool = TavilySearch(max_results=5)

@tool
def scrape_webpages(urls:List[str]):
    """요청한 웹 페이지 목록을 크롤링하여 본문 텍스트를 추출합니다."""
    loader = WebBaseLoader(
        urls,
        header_template={
            "User-Agent":(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
                "AppleWebKit/537.36 (KHTML,like Gecko)"
                "Chrome/122.0.0.0 Safari/537.36"
            )})
    docs = loader.load()
    return "\n\n".join(
        [
            f'<문서 제목="{doc.metadata.get("title","")}">\n{doc.page_content}\n</문서>'
            for doc in docs
        ]
    )


In [6]:
# 3) 문서팀 도구
WORKING_DIRECTORY = Path.cwd()

In [7]:
@tool
def create_outline(
    points: Annotated[List[str], "섹션/소제목 리스트"],
    file_name: Annotated[str, "개요를 저장할 파일 경로(상대/절대)"]):
    """개요(아웃라인)를 파일로 저장힙니다."""
    path = (WORKING_DIRECTORY / file_name)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for i, p in enumerate(points):
            f.write(f"{i + 1}. {p}\n")
    return f"개요 저장 완료: {path}"

In [8]:
# 문서 읽기 : read_document()
@tool
def read_document(
    file_name: Annotated[str, "읽을 파일 경로"],
    start: Annotated[Optional[int], "시작 줄(기본 0)"] = None,
    end: Annotated[Optional[int], "끝 줄(기본 None)"] = None):
    """지정한 텍스트 문서를 읽어 반환합니다(부분 범위 가능)."""
    path = (WORKING_DIRECTORY / file_name)
    with path.open("r", encoding="utf-8") as f:
        lines = f.readlines()
    start = 0 if start is None else start
    return "".join(lines[start:end])

In [10]:
# 본문 저장 : write_document()
@tool
def write_document(
    content: Annotated[str, "파일에 저장할 텍스트 콘텐츠"],
    file_name: Annotated[str, "저장할 파일 경로"]):
    """텍스트 문서를 새로 작성/덮어씁니다."""
    path = (WORKING_DIRECTORY / file_name)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        f.write(content)
    return f"문서 저장 완료: {path}"

In [11]:
# 부분 수정 : edit_document()
@tool
def edit_document(
    file_name: Annotated[str, "수정할 파일 경로"],
    inserts: Annotated[
        Dict[int, str],
        "삽입할 위치(1부터 시작)와 텍스트의 딕셔너리",
    ]):
    """지정한 줄 번호 위치에 문장을 삽입합니다."""
    path = (WORKING_DIRECTORY / file_name)
    with path.open("r", encoding="utf-8") as f:
        lines = f.readlines()

    for line_no, text in sorted(inserts.items()):
        if 1 <= line_no <= len(lines) + 1:
            lines.insert(line_no - 1, text + "\n")
        else:
            return f"오류:줄 번호 {line_no} 범위를 벗어났습니다."

    with path.open("w", encoding="utf-8") as f:
        f.writelines(lines)
    return f"문서 수정/저장 완료: {path}"

In [12]:
# 로컬 파이썬 실행 (차트 등)
repl = PythonREPL

@tool
def python_repl_tool(
    code: Annotated[str, "차트 생성 등 실행할 파이썬 코드(출력은 print로 표시)"],
):
    """파이썬 코드를 실행합니다. 결과를 보려면 print(...)를 사용하세요."""
    try:
        result = repl.run(code)
    except BaseException as e:
        return f"실행 실패: {repr(e)}"
    return f"성공적으로 실행했습니다:\n'''python\n{code}\n'''\nStdout: {result}"

In [ ]:
# 4) "감독자(supervisor)" 노드 팩토리
# - 멤버 이름 목록을 받아, 다음에 호출할 워커를 LLM이 결정
# - Pydantic 스키마로 구조화된 출력 사용
def make_supervisor_node(llm, members: List[str]): # 리서치 감독자라면, members=["search", "web_scrapter"] 임
    options = ["FINISH", *members] # ["FINISH", "search", "web_scraper"]

    system_prompt = (
        "당신은 다음 워커들을 관리하는 감독자입니다: "
        f"{members}."
        "사용자 요청과 현재까지의 대화 메시지를 보고 "
        "다음에 실행할 워커를 한 명만 선택하세요. "
        "모든 작업이 끝났다면 FINISH를 선택하세요. "
    )

    class Router(BaseModel):
        next: Literal[*options] = Field(..., description="다음 노드명 또는 FINISH")